# 😷 Face Mask Detection — Classification Model

**Dataset:** [Face Mask Detection (Kaggle)](https://www.kaggle.com/datasets/andrewmvd/face-mask-detection)

This notebook:
1. Downloads the dataset from Kaggle
2. Parses Pascal VOC XML annotations to crop individual faces
3. Splits data into Train / Validation / Test sets
4. Fine-tunes **MobileNetV2** for 3-class classification:
   - `with_mask`
   - `without_mask`
   - `mask_weared_incorrect`
5. Evaluates accuracy and generates plots (training curves, confusion matrix, sample predictions)

> **Runtime:** Go to *Runtime → Change runtime type → GPU (T4)* for faster training.

## 1 · Install & Import Libraries

In [ ]:
!pip install -q kaggle

import os, zipfile, glob, random, warnings
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## 2 · Download Dataset from Kaggle

Upload your `kaggle.json` API key when prompted.  
*(Get it from [kaggle.com → Account → Create New Token](https://www.kaggle.com/settings/account))*

In [ ]:
from google.colab import files

# Upload kaggle.json
uploaded = files.upload()

# Set up Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
for fn in uploaded.keys():
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
        f.write(uploaded[fn])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle credentials configured.')

In [ ]:
# Download and extract the dataset
!kaggle datasets download -d andrewmvd/face-mask-detection -p /content/dataset --unzip
print('\n✅ Dataset downloaded and extracted.')
!ls /content/dataset

## 3 · Parse XML Annotations & Crop Faces

In [ ]:
# Locate images and annotations
IMG_DIR = '/content/dataset/images'
ANN_DIR = '/content/dataset/annotations'

# Verify paths exist; some extractions nest differently
if not os.path.isdir(IMG_DIR):
    # Try alternate structure
    for root, dirs, _files in os.walk('/content/dataset'):
        if 'images' in dirs:
            IMG_DIR = os.path.join(root, 'images')
        if 'annotations' in dirs:
            ANN_DIR = os.path.join(root, 'annotations')

print(f'Images dir : {IMG_DIR}  ({len(os.listdir(IMG_DIR))} files)')
print(f'Annot. dir : {ANN_DIR}  ({len(os.listdir(ANN_DIR))} files)')

In [ ]:
def parse_annotation(xml_path):
    """Parse a Pascal VOC XML file and return list of (label, bbox) tuples."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    objects = []
    for obj in root.findall('object'):
        label = obj.find('name').text.strip()
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        objects.append({'label': label, 'bbox': (xmin, ymin, xmax, ymax)})
    return filename, objects

# Parse all annotations
records = []
xml_files = sorted(glob.glob(os.path.join(ANN_DIR, '*.xml')))
for xf in xml_files:
    fname, objs = parse_annotation(xf)
    for o in objs:
        records.append({'image': fname, 'label': o['label'],
                        'xmin': o['bbox'][0], 'ymin': o['bbox'][1],
                        'xmax': o['bbox'][2], 'ymax': o['bbox'][3]})

df = pd.DataFrame(records)
print(f'Total face annotations: {len(df)}')
print(df['label'].value_counts())

In [ ]:
# Crop face regions and save into class-wise folders
CROP_DIR = '/content/cropped_faces'
IMG_SIZE = 128  # resize crops to 128x128

for label in df['label'].unique():
    os.makedirs(os.path.join(CROP_DIR, label), exist_ok=True)

crop_count = 0
for idx, row in df.iterrows():
    img_path = os.path.join(IMG_DIR, row['image'])
    if not os.path.exists(img_path):
        continue
    try:
        img = Image.open(img_path).convert('RGB')
        w, h = img.size
        # Clamp bbox to image dimensions
        xmin = max(0, row['xmin'])
        ymin = max(0, row['ymin'])
        xmax = min(w, row['xmax'])
        ymax = min(h, row['ymax'])
        if xmax <= xmin or ymax <= ymin:
            continue
        crop = img.crop((xmin, ymin, xmax, ymax)).resize((IMG_SIZE, IMG_SIZE))
        save_path = os.path.join(CROP_DIR, row['label'], f'face_{idx}.jpg')
        crop.save(save_path)
        crop_count += 1
    except Exception as e:
        pass

print(f'\n✅ Cropped {crop_count} face images.')
for label in sorted(os.listdir(CROP_DIR)):
    n = len(os.listdir(os.path.join(CROP_DIR, label)))
    print(f'  {label:30s} → {n} images')

## 4 · Exploratory Data Analysis

In [ ]:
# ---- Class Distribution ----
class_counts = df['label'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#f39c12']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(class_counts.index, class_counts.values, color=colors)
axes[0].set_title('Class Distribution (Bar)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (lbl, cnt) in enumerate(class_counts.items()):
    axes[0].text(i, cnt + 20, str(cnt), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, textprops={'fontsize': 11})
axes[1].set_title('Class Distribution (Pie)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ---- Sample Cropped Faces ----
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for row_idx, label in enumerate(sorted(os.listdir(CROP_DIR))):
    folder = os.path.join(CROP_DIR, label)
    imgs = random.sample(os.listdir(folder), min(5, len(os.listdir(folder))))
    for col_idx, img_name in enumerate(imgs):
        img = Image.open(os.path.join(folder, img_name))
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].set_title(label, fontsize=10)
        axes[row_idx][col_idx].axis('off')
fig.suptitle('Sample Cropped Faces per Class', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5 · Prepare Data Generators (Train / Val / Test)

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

SPLIT_DIR = '/content/split_data'
CLASSES = sorted(os.listdir(CROP_DIR))
print('Classes:', CLASSES)

for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        os.makedirs(os.path.join(SPLIT_DIR, split, cls), exist_ok=True)

# Split: 70% train, 15% val, 15% test
for cls in CLASSES:
    folder = os.path.join(CROP_DIR, cls)
    all_imgs = os.listdir(folder)
    train_imgs, temp_imgs = train_test_split(all_imgs, test_size=0.30, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)

    for img in train_imgs:
        shutil.copy(os.path.join(folder, img), os.path.join(SPLIT_DIR, 'train', cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(folder, img), os.path.join(SPLIT_DIR, 'val', cls, img))
    for img in test_imgs:
        shutil.copy(os.path.join(folder, img), os.path.join(SPLIT_DIR, 'test', cls, img))

# Summary
for split in ['train', 'val', 'test']:
    total = sum(len(os.listdir(os.path.join(SPLIT_DIR, split, c))) for c in CLASSES)
    print(f'{split:5s}: {total} images')

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('\nClass indices:', train_gen.class_indices)

## 6 · Build the Model (MobileNetV2 Transfer Learning)

In [ ]:
NUM_CLASSES = len(CLASSES)

# Load MobileNetV2 without top layers
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # Freeze base layers

# Build classifier head
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7 · Train the Model

In [ ]:
EPOCHS = 15

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)
print('\n✅ Training complete.')

## 8 · Fine-Tune (Unfreeze Top Layers)

In [ ]:
# Unfreeze the last 30 layers of MobileNetV2 for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

FINE_TUNE_EPOCHS = 10

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks
)
print('\n✅ Fine-tuning complete.')

## 9 · Evaluate on Test Set

In [ ]:
# Evaluate
test_loss, test_accuracy = model.evaluate(test_gen)
print(f'\n📊 Test Loss    : {test_loss:.4f}')
print(f'📊 Test Accuracy : {test_accuracy * 100:.2f}%')

In [ ]:
# Predictions & Classification Report
test_gen.reset()
y_pred_probs = model.predict(test_gen)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

class_labels = list(test_gen.class_indices.keys())

print('\n' + '='*60)
print('           CLASSIFICATION REPORT')
print('='*60)
print(classification_report(y_true, y_pred, target_names=class_labels))

## 10 · Plots & Visualizations

In [ ]:
# ---- Training & Fine-Tuning Curves ----
# Combine history from both phases
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']
epochs_range = range(1, len(acc) + 1)
ft_start = len(history.history['accuracy']) + 1  # fine-tune start epoch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
ax1.plot(epochs_range, acc, 'b-o', label='Train Accuracy', markersize=4)
ax1.plot(epochs_range, val_acc, 'r-o', label='Val Accuracy', markersize=4)
ax1.axvline(x=ft_start, color='green', linestyle='--', label='Fine-Tune Start')
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(epochs_range, loss, 'b-o', label='Train Loss', markersize=4)
ax2.plot(epochs_range, val_loss, 'r-o', label='Val Loss', markersize=4)
ax2.axvline(x=ft_start, color='green', linestyle='--', label='Fine-Tune Start')
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Confusion Matrix ----
cm = confusion_matrix(y_true, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax1)
ax1.set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')

# Normalized (percentage)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Oranges',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax2)
ax2.set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ---- Per-Class Accuracy Bar Chart ----
per_class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(8, 5))
bars = plt.bar(class_labels, per_class_acc * 100, color=['#2ecc71', '#e74c3c', '#f39c12'])
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 105)
for bar, acc_val in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc_val*100:.1f}%', ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Sample Predictions ----
test_gen.reset()
x_batch, y_batch = next(test_gen)
preds = model.predict(x_batch)

fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for i, ax in enumerate(axes.flat):
    if i >= len(x_batch):
        ax.axis('off')
        continue
    ax.imshow(x_batch[i])
    true_label = class_labels[np.argmax(y_batch[i])]
    pred_label = class_labels[np.argmax(preds[i])]
    confidence = np.max(preds[i]) * 100
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

fig.suptitle('Sample Test Predictions (Green=Correct, Red=Wrong)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11 · Save the Model

In [ ]:
# Save the trained model
model.save('/content/face_mask_classifier.h5')
print('✅ Model saved as face_mask_classifier.h5')

# Download from Colab (optional)
from google.colab import files
files.download('/content/face_mask_classifier.h5')

## 12 · Summary

| Metric | Value |
|--------|-------|
| **Model** | MobileNetV2 (Transfer Learning + Fine-Tuning) |
| **Classes** | `with_mask`, `without_mask`, `mask_weared_incorrect` |
| **Input size** | 128 × 128 × 3 |
| **Test Accuracy** | *(printed above)* |

### Key Takeaways
- Transfer learning with **MobileNetV2** provides strong accuracy on this relatively small dataset.
- Data augmentation helps generalize despite class imbalance.
- The **`mask_weared_incorrect`** class is the hardest due to its small sample count and visual similarity to `with_mask`.